# Notebook 02: Construct the Analytic Sample and Operational Targets

## Purpose

Validate Notebook 01 outputs, document participant flow and missingness, quantify target prevalence and concordance, construct the four joint target groups, and save the labelled complete-case dataset used downstream.

## Inputs

- `data/processed/nhanes_diabetes_analysis_base.csv`
- `data/processed/nhanes_diabetes_complete_case.csv`
- `data/processed/sample_metadata.json`

## Outputs

- `data/processed/nhanes_diabetes_complete_case_labeled.csv`
- Participant-flow, prevalence, concordance, and joint-group tables
- `data/processed/sample_and_label_metadata.json`

## Dependencies

Run Notebook 01 first. Notebooks 03--09 depend on the labelled complete-case dataset produced here.

> **Repository policy:** Notebook outputs and execution counts are cleared in the public source files. Run the notebooks in the documented order to regenerate all results.

## 1. Setup

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 150)


## 2. Define project paths

In [ ]:
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR = PROJECT_DIR / "outputs"
TABLE_DIR = OUTPUT_DIR / "tables"

TABLE_DIR.mkdir(parents=True, exist_ok=True)

analysis_base_path = PROCESSED_DIR / "nhanes_diabetes_analysis_base.csv"
complete_case_path = PROCESSED_DIR / "nhanes_diabetes_complete_case.csv"
metadata_path = PROCESSED_DIR / "sample_metadata.json"

print("Project directory:", PROJECT_DIR)
print("Analysis base:", analysis_base_path)
print("Complete-case data:", complete_case_path)


## 3. Load processed datasets and metadata

In [ ]:
analysis_base = pd.read_csv(analysis_base_path)
complete_case = pd.read_csv(complete_case_path)

with metadata_path.open("r", encoding="utf-8") as file:
    sample_metadata = json.load(file)

print("Analysis-base shape:", analysis_base.shape)
print("Complete-case shape:", complete_case.shape)
print("Metadata:")
print(sample_metadata)


## 4. Validate the loaded data

In [ ]:
required_base_columns = {
    "id",
    "age",
    "sex",
    "race_ethnicity",
    "pregnancy_status",
    "confirmed_current_pregnancy",
    "primary_sample_eligible",
    "income_poverty_ratio",
    "bmi",
    "currently_insured",
    "insurance_history",
    "hba1c",
    "self_reported_prior_diagnosis",
    "current_hba1c_ge_6_5",
    "blood_test_past_3y",
    "insulin_now",
    "diabetes_pills_now",
    "any_diabetes_medication",
    "exam_status",
    "mec_exam_weight",
    "survey_stratum",
    "survey_psu",
}

required_complete_case_columns = {
    "id",
    "age",
    "sex",
    "race_ethnicity",
    "pregnancy_status",
    "confirmed_current_pregnancy",
    "primary_sample_eligible",
    "income_poverty_ratio",
    "bmi",
    "insurance_history",
    "self_reported_prior_diagnosis",
    "current_hba1c_ge_6_5",
    "exam_status",
    "mec_exam_weight",
    "survey_stratum",
    "survey_psu",
}

missing_base_columns = required_base_columns.difference(analysis_base.columns)
missing_complete_columns = required_complete_case_columns.difference(
    complete_case.columns
)

if missing_base_columns:
    raise KeyError(
        f"Analysis base is missing expected columns: {sorted(missing_base_columns)}"
    )

if missing_complete_columns:
    raise KeyError(
        "Complete-case data are missing expected columns: "
        f"{sorted(missing_complete_columns)}"
    )

if not analysis_base["id"].is_unique:
    raise ValueError("The analysis base contains duplicate participant IDs.")

if not complete_case["id"].is_unique:
    raise ValueError("The complete-case data contain duplicate participant IDs.")

if not set(complete_case["id"]).issubset(set(analysis_base["id"])):
    raise ValueError(
        "The complete-case participant IDs are not a subset of the analysis base."
    )

# Guard against SAS/XPT numeric missing-value placeholders that can look like
# extremely small positive values rather than NaN. Notebook 01 should remove
# these before constructing the complete-case sample.
placeholder_check_columns = [
    "income_poverty_ratio",
    "bmi",
    "hba1c",
    "mec_exam_weight",
]

for dataset_name, dataframe in {
    "analysis base": analysis_base,
    "complete-case data": complete_case,
}.items():
    for column in placeholder_check_columns:
        placeholder_mask = (
            dataframe[column].notna()
            & dataframe[column].gt(0)
            & dataframe[column].lt(1e-50)
        )

        if placeholder_mask.any():
            raise ValueError(
                f"The {dataset_name} still contains an SAS/XPT numeric "
                f"missing-value placeholder in {column}. Rerun the corrected "
                "Notebook 01 before continuing."
            )

target_columns = [
    "self_reported_prior_diagnosis",
    "current_hba1c_ge_6_5",
]

predictor_columns = [
    "age",
    "sex",
    "race_ethnicity",
    "income_poverty_ratio",
    "bmi",
    "insurance_history",
]

survey_columns = [
    "exam_status",
    "mec_exam_weight",
    "survey_stratum",
    "survey_psu",
]

required_model_columns = predictor_columns + target_columns
required_analysis_columns = required_model_columns + survey_columns

remaining_missing = complete_case[required_analysis_columns].isna().sum()
if remaining_missing.sum() != 0:
    raise ValueError(
        "The complete-case dataset still contains missing values in required "
        f"analysis columns:\n{remaining_missing[remaining_missing > 0]}"
    )

for target in target_columns:
    observed_values = set(complete_case[target].dropna().unique())
    if not observed_values.issubset({0, 1, 0.0, 1.0}):
        raise ValueError(
            f"{target} contains values other than 0 and 1: {observed_values}"
        )


if complete_case["confirmed_current_pregnancy"].ne(0).any():
    raise ValueError(
        "The complete-case dataset contains a confirmed current pregnancy."
    )

if not complete_case["primary_sample_eligible"].eq(1).all():
    raise ValueError(
        "The complete-case dataset contains a participant who is not eligible "
        "for the primary analysis."
    )

if int(analysis_base["confirmed_current_pregnancy"].sum()) != int(
    sample_metadata["n_confirmed_current_pregnancy_excluded"]
):
    raise ValueError(
        "The confirmed-pregnancy count does not match Notebook 01 metadata."
    )

if int(analysis_base["primary_sample_eligible"].sum()) != int(
    sample_metadata["n_primary_eligible_adults"]
):
    raise ValueError(
        "The primary-eligibility count does not match Notebook 01 metadata."
    )

if not complete_case["exam_status"].eq(2).all():
    raise ValueError(
        "The complete-case dataset contains participants who were not examined "
        "in the MEC."
    )

if not complete_case["mec_exam_weight"].gt(0).all():
    raise ValueError(
        "The complete-case dataset contains a non-positive MEC examination weight."
    )

if len(analysis_base) != int(sample_metadata["n_adults"]):
    raise ValueError(
        "The analysis-base row count does not match the metadata from Notebook 01."
    )

if len(complete_case) != int(sample_metadata["n_complete_case"]):
    raise ValueError(
        "The complete-case row count does not match the metadata from Notebook 01."
    )

print("Validation passed.")


## 5. Participant flow

This table documents nested stages from the merged NHANES files to the matched
complete-case sample used for both targets. Confirmed current pregnancy is
shown as an explicit clinical eligibility exclusion.

In [ ]:
n_all_ages = int(sample_metadata["n_merged_all_ages"])
n_adults = len(analysis_base)
n_confirmed_pregnancy = int(
    analysis_base["confirmed_current_pregnancy"].sum()
)
n_primary_eligible = int(
    analysis_base["primary_sample_eligible"].sum()
)

eligible_mask = analysis_base["primary_sample_eligible"].eq(1)
examined_eligible_mask = (
    eligible_mask
    & analysis_base["exam_status"].eq(2)
)
both_targets_mask = (
    examined_eligible_mask
    & analysis_base[target_columns].notna().all(axis=1)
)

n_examined_eligible = int(examined_eligible_mask.sum())
n_both_targets = int(both_targets_mask.sum())
n_complete = len(complete_case)

participant_flow = pd.DataFrame(
    {
        "stage": [
            "Merged NHANES participants, all ages",
            "Adults aged 18 years or older",
            "Adults eligible after excluding confirmed current pregnancy",
            "Eligible adults examined in the MEC",
            "Eligible examined adults with both operational targets observed",
            (
                "Eligible adults with both targets, all model predictors, "
                "and survey fields observed"
            ),
        ],
        "n": [
            n_all_ages,
            n_adults,
            n_primary_eligible,
            n_examined_eligible,
            n_both_targets,
            n_complete,
        ],
    }
)

participant_flow["removed_since_previous_stage"] = (
    participant_flow["n"].shift(1) - participant_flow["n"]
)
participant_flow.loc[0, "removed_since_previous_stage"] = np.nan

participant_flow["share_of_adult_analysis_base"] = (
    participant_flow["n"] / n_adults
)
participant_flow.loc[0, "share_of_adult_analysis_base"] = np.nan

if participant_flow["n"].is_monotonic_decreasing is False:
    raise ValueError("Participant-flow stages are not nested.")

if n_adults - n_primary_eligible != n_confirmed_pregnancy:
    raise ValueError(
        "The pregnancy exclusion does not reconcile with primary eligibility."
    )

participant_flow.to_csv(
    TABLE_DIR / "participant_flow.csv",
    index=False,
)

participant_flow


## 6. Missingness relevant to sample construction

This table records missingness in the adult analysis base. Pregnancy status is
not used as a complete-case requirement because missingness is often
structural. Confirmed current pregnancy is handled through the explicit
eligibility indicator rather than through missing-value deletion.

Interpretation of how included and excluded participants differ is deferred to
Notebook 03.

In [ ]:
sample_construction_columns = [
    "age",
    "sex",
    "race_ethnicity",
    "pregnancy_status",
    "confirmed_current_pregnancy",
    "primary_sample_eligible",
    "income_poverty_ratio",
    "bmi",
    "insurance_history",
    "self_reported_prior_diagnosis",
    "hba1c",
    "current_hba1c_ge_6_5",
]

missingness = (
    analysis_base[sample_construction_columns]
    .isna()
    .agg(["sum", "mean"])
    .T
    .rename(
        columns={
            "sum": "missing_count",
            "mean": "missing_share",
        }
    )
    .sort_values("missing_share", ascending=False)
)

missingness["missing_count"] = missingness["missing_count"].astype(int)

missingness.to_csv(
    TABLE_DIR / "sample_construction_missingness.csv"
)

missingness


## 7. Target counts and prevalence

The unweighted values describe the complete-case analytic sample. The MEC-weighted values are also calculated within this complete-case sample. Because predictor missingness caused additional exclusions, these weighted values should be described as **MEC-weighted complete-case point estimates**, not as fully design-corrected population prevalence estimates. Formal survey-design standard errors are not calculated here.

In [ ]:
def weighted_mean(values: pd.Series, weights: pd.Series) -> float:
    mask = values.notna() & weights.notna() & (weights > 0)

    if mask.sum() == 0:
        return np.nan

    return float(
        np.average(
            values.loc[mask].astype(float),
            weights=weights.loc[mask].astype(float),
        )
    )


prevalence_rows = []

for target in target_columns:
    prevalence_rows.append(
        {
            "target": target,
            "analytic_sample_n": int(complete_case[target].notna().sum()),
            "positive_n": int((complete_case[target] == 1).sum()),
            "negative_n": int((complete_case[target] == 0).sum()),
            "unweighted_prevalence": float(complete_case[target].mean()),
            "mec_weighted_complete_case_prevalence": weighted_mean(
                complete_case[target],
                complete_case["mec_exam_weight"],
            ),
        }
    )

prevalence_table = pd.DataFrame(prevalence_rows)

prevalence_table.to_csv(
    TABLE_DIR / "target_prevalence.csv",
    index=False,
)

prevalence_table


## 8. Unweighted agreement between the two operational labels

The count table is the central starting point for the later descriptive analysis. The row-normalised table answers: within each prior-diagnosis category, what proportion currently has HbA1c below or above 6.5%?

In [ ]:
label_concordance_counts = pd.crosstab(
    complete_case["self_reported_prior_diagnosis"],
    complete_case["current_hba1c_ge_6_5"],
    rownames=["Self-reported prior diagnosis"],
    colnames=["Current HbA1c ≥ 6.5%"],
    margins=True,
)

label_concordance_row_proportions = pd.crosstab(
    complete_case["self_reported_prior_diagnosis"],
    complete_case["current_hba1c_ge_6_5"],
    rownames=["Self-reported prior diagnosis"],
    colnames=["Current HbA1c ≥ 6.5%"],
    normalize="index",
)

label_concordance_counts.to_csv(
    TABLE_DIR / "label_concordance_counts.csv"
)

label_concordance_row_proportions.to_csv(
    TABLE_DIR / "label_concordance_row_proportions.csv"
)

label_concordance_counts


In [ ]:
label_concordance_row_proportions


## 9. MEC-weighted agreement between the labels

These are MEC-weighted shares within the complete-case analytic sample, not raw participant counts and not fully design-corrected population estimates. Formal survey-design uncertainty is deferred.

In [ ]:
weighted_concordance_sums = pd.pivot_table(
    complete_case,
    index="self_reported_prior_diagnosis",
    columns="current_hba1c_ge_6_5",
    values="mec_exam_weight",
    aggfunc="sum",
    fill_value=0,
)

weighted_concordance_total_shares = (
    weighted_concordance_sums
    / weighted_concordance_sums.to_numpy().sum()
)

weighted_concordance_row_shares = weighted_concordance_sums.div(
    weighted_concordance_sums.sum(axis=1),
    axis=0,
)

weighted_concordance_sums.to_csv(
    TABLE_DIR / "label_concordance_weighted_sums.csv"
)

weighted_concordance_total_shares.to_csv(
    TABLE_DIR / "label_concordance_weighted_total_shares.csv"
)

weighted_concordance_row_shares.to_csv(
    TABLE_DIR / "label_concordance_weighted_row_shares.csv"
)

weighted_concordance_total_shares


## 10. Create the four joint label groups

Neutral names are used because neither operational label is assumed to be the definitive clinical truth.

In [ ]:
conditions = [
    (
        (complete_case["self_reported_prior_diagnosis"] == 0)
        & (complete_case["current_hba1c_ge_6_5"] == 0)
    ),
    (
        (complete_case["self_reported_prior_diagnosis"] == 0)
        & (complete_case["current_hba1c_ge_6_5"] == 1)
    ),
    (
        (complete_case["self_reported_prior_diagnosis"] == 1)
        & (complete_case["current_hba1c_ge_6_5"] == 0)
    ),
    (
        (complete_case["self_reported_prior_diagnosis"] == 1)
        & (complete_case["current_hba1c_ge_6_5"] == 1)
    ),
]

label_names = [
    "No prior diagnosis / HbA1c below 6.5",
    "No prior diagnosis / HbA1c at least 6.5",
    "Prior diagnosis / HbA1c below 6.5",
    "Prior diagnosis / HbA1c at least 6.5",
]

joint_label_codes = [
    "D0_H0",
    "D0_H1",
    "D1_H0",
    "D1_H1",
]

complete_case = complete_case.copy()

complete_case["label_group"] = np.select(
    conditions,
    label_names,
    default=pd.NA,
)

complete_case["joint_label_code"] = np.select(
    conditions,
    joint_label_codes,
    default=pd.NA,
)

if complete_case["label_group"].isna().any():
    raise ValueError("At least one participant could not be assigned to a label group.")

label_group_counts = (
    complete_case["label_group"]
    .value_counts()
    .reindex(label_names)
    .rename("n")
    .to_frame()
)

label_group_counts["share"] = (
    label_group_counts["n"] / len(complete_case)
)

if int(label_group_counts["n"].sum()) != len(complete_case):
    raise ValueError(
        "The four joint label-group counts do not sum to the complete-case sample."
    )

N_SPLITS = 5
minimum_joint_group_n = int(label_group_counts["n"].min())

if minimum_joint_group_n < N_SPLITS:
    raise ValueError(
        "The smallest joint label group is too small for five-fold stratification."
    )

label_group_counts.to_csv(
    TABLE_DIR / "label_group_counts.csv"
)

label_group_counts


## 11. Save the labelled complete-case dataset

The participant-level file remains in `data/processed` and should stay excluded from GitHub. Later notebooks can use `joint_label_code` to construct identical stratified folds for both prediction targets. The codes use the stable format `D0_H0`, `D0_H1`, `D1_H0`, and `D1_H1`, so CSV loading cannot remove leading zeros.

In [ ]:
labelled_complete_case_path = (
    PROCESSED_DIR / "nhanes_diabetes_complete_case_labeled.csv"
)

complete_case.to_csv(
    labelled_complete_case_path,
    index=False,
)

print("Saved labelled complete-case dataset to:")
print(labelled_complete_case_path)
print("Shape:", complete_case.shape)


## 12. Final checkpoint

In [ ]:
final_checkpoint = {
    "analysis_base_n": int(len(analysis_base)),
    "complete_case_n": int(len(complete_case)),
    "confirmed_current_pregnancy_excluded_n": int(
        analysis_base["confirmed_current_pregnancy"].sum()
    ),
    "primary_sample_eligible_adults_n": int(
        analysis_base["primary_sample_eligible"].sum()
    ),
    "confirmed_current_pregnancy_in_complete_case_n": int(
        complete_case["confirmed_current_pregnancy"].sum()
    ),
    "complete_case_share_of_adult_base": float(
        len(complete_case) / len(analysis_base)
    ),
    "self_reported_prior_diagnosis_positive_n": int(
        (complete_case["self_reported_prior_diagnosis"] == 1).sum()
    ),
    "current_hba1c_ge_6_5_positive_n": int(
        (complete_case["current_hba1c_ge_6_5"] == 1).sum()
    ),
    "no_prior_diagnosis_hba1c_ge_6_5_n": int(
        (
            (complete_case["self_reported_prior_diagnosis"] == 0)
            & (complete_case["current_hba1c_ge_6_5"] == 1)
        ).sum()
    ),
    "prior_diagnosis_hba1c_below_6_5_n": int(
        (
            (complete_case["self_reported_prior_diagnosis"] == 1)
            & (complete_case["current_hba1c_ge_6_5"] == 0)
        ).sum()
    ),
    "remaining_missing_required_analysis_values": int(
        complete_case[required_analysis_columns].isna().sum().sum()
    ),
    "all_complete_cases_primary_sample_eligible": bool(
        complete_case["primary_sample_eligible"].eq(1).all()
    ),
    "all_complete_cases_examined": bool(
        complete_case["exam_status"].eq(2).all()
    ),
    "all_complete_case_mec_weights_positive": bool(
        complete_case["mec_exam_weight"].gt(0).all()
    ),
    "joint_label_counts": (
        complete_case["joint_label_code"]
        .value_counts()
        .sort_index()
        .to_dict()
    ),
    "minimum_joint_label_group_n": minimum_joint_group_n,
    "recommended_cross_validation_folds": N_SPLITS,
    "weighted_estimate_scope": "MEC-weighted complete-case point estimates",
}

with (
    PROCESSED_DIR / "sample_and_label_metadata.json"
).open("w", encoding="utf-8") as file:
    json.dump(final_checkpoint, file, indent=2)

final_checkpoint


## Completion criteria

- Participant identifiers are unique and nested correctly.
- Both operational targets are binary and complete in the analytic sample.
- Joint target groups and prevalence estimates reconcile with the saved metadata.